# GPU Benchmark: Setpoint / Schedule Optimization

Companion to `gpu_benchmark_estimation.ipynb`, focused on the **optimizer**: given a
calibrated model, find the control schedule (here: space-heater water flow) that
minimizes energy cost subject to comfort constraints. Runs directly in
[Google Colab](https://colab.research.google.com); select a GPU runtime for the
CPU-vs-GPU comparison.

## What is measured

Twin4Build's tensors are CPU-only today, so as in the estimation notebook we
measure (A) the **real optimizer pipeline** on the current runtime for a hard
baseline, and (B) the optimizer's **dominant kernels** on CPU vs GPU:

- **O1 — gradient through the rollout**: the fast control objective
  differentiates the loss (energy + cost + comfort penalties) through the full
  simulation horizon w.r.t. the decision schedule. One evaluation = forward rollout
  + backward pass. This is the optimizer's per-iteration cost.
- **O2 — multi-scenario evaluation**: the same rollout batched over `B` weather /
  price / occupancy scenarios (stochastic MPC, robust optimization) or `B`
  random restarts of a non-convex schedule search.

The decision variable dimension is the schedule itself (one value per timestep),
so unlike estimation the *gradient* here has hundreds of entries — reverse-mode
autodiff through the rollout gives all of them in one backward pass, on either
device.

In [ ]:
# Colab: set T4B_REF to refs/heads/<branch>, refs/tags/<tag>, or a commit SHA.
# (GitHub archive zip — works with '/' in branch names. Skip this cell locally.)
T4B_REF = "refs/heads/fix/full-workflow-portable-data"
!pip install -q --upgrade "https://github.com/JBjoernskov/Twin4Build/archive/{T4B_REF}.zip"

import twin4build as tb

# --- Setup (Colab-aware) ---------------------------------------------------

import datetime
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dateutil import tz

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
print(f"torch {torch.__version__}")
print(f"CPU threads: {torch.get_num_threads()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available -- kernel benchmarks will run on CPU only.")
    print("(In Colab: Runtime > Change runtime type > T4 GPU)")


## Part A — Real optimizer pipeline on the current (CPU) runtime

A compact but complete optimal-control problem: a thermal zone with a hydronic
space heater (bidirectionally coupled — heater power heats the zone, zone
temperature sets the heater's return conditions), a time-varying electricity
price, and comfort setpoint bands. The optimizer's decision variables are the
water-flow schedule values (one per timestep, 36 over a day at 40-minute steps);
the objective minimizes heater energy and cost subject to the comfort band as
inequality constraints.

This is the same model the optimizer's fast-objective regression test uses, so the
timing reflects the production code path (`options={"fast": True}` is the default:
composed one-step map + autodiff).

In [ ]:
model = tb.Model(id="gpu_benchmark_optimizer")

building_space = tb.BuildingSpaceThermalTorchSystem(
    C_air=2e6, C_wall=1e7, C_boundary=8e5,
    R_out=0.005, R_in=0.005, R_boundary=1e4,
    f_wall=0, f_air=0, Q_occ_gain=100.0,
    CO2_occ_gain=0.004, CO2_start=400.0,
    infiltrationRate=0.0, airVolume=100.0,
    id="BuildingSpace",
)
space_heater = tb.SpaceHeaterTorchSystem(
    Q_flow_nominal_sh=2000.0, T_a_nominal_sh=60.0, T_b_nominal_sh=30.0,
    TAir_nominal_sh=21.0, thermalMassHeatCapacity=5e5, nelements=3,
    id="SpaceHeater",
)

const = lambda v, i: tb.ScheduleSystem(
    weekDayRulesetDict={"ruleset_default_value": v}, id=i
)
occupancy = const(0, "Occupancy")
solar = const(0.0, "Solar")
supply_flow = const(0.0, "SupplyFlow")
exhaust_flow = const(0.0, "ExhaustFlow")
supply_air_temp = const(20.0, "SupplyAirTemp")
supply_water_temp = const(60.0, "SupplyWaterTemp")
outdoor_temp = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 10.0,
        "ruleset_start_minute": [0, 0], "ruleset_end_minute": [0, 0],
        "ruleset_start_hour": [0, 12], "ruleset_end_hour": [12, 24],
        "ruleset_value": [5.0, 12.0],
    },
    id="OutdoorTemperature",
)

mf = 2000.0 / 4180 / (60.0 - 30.0)  # nominal water mass flow [kg/s]
waterflow = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 0,
        "ruleset_start_minute": [0], "ruleset_end_minute": [0],
        "ruleset_start_hour": [8], "ruleset_end_hour": [16],
        "ruleset_value": [mf],
    },
    id="Waterflow",
)
price = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 0.5,
        "ruleset_start_minute": [0, 0], "ruleset_end_minute": [0, 0],
        "ruleset_start_hour": [6, 17], "ruleset_end_hour": [9, 20],
        "ruleset_value": [2.0, 2.5],
    },
    id="Price",
)
costs = tb.ScalarProductSystem(scale_factor=2400 / 3600 / 1000, id="Costs")

model.add_connection(occupancy, building_space, "scheduleValue", "numberOfPeople")
model.add_connection(outdoor_temp, building_space, "scheduleValue", "outdoorTemperature")
model.add_connection(solar, building_space, "scheduleValue", "globalIrradiation")
model.add_connection(supply_flow, building_space, "scheduleValue", "supplyAirFlowRate")
model.add_connection(exhaust_flow, building_space, "scheduleValue", "exhaustAirFlowRate")
model.add_connection(supply_air_temp, building_space, "scheduleValue", "supplyAirTemperature")
model.add_connection(supply_water_temp, space_heater, "scheduleValue", "supplyWaterTemperature")
model.add_connection(waterflow, space_heater, "scheduleValue", "waterFlowRate")
model.add_connection(building_space, space_heater, "indoorTemperature", "indoorTemperature")
model.add_connection(space_heater, building_space, "Power", "heatGain")
model.add_connection(space_heater, costs, "Power", "input_1")
model.add_connection(price, costs, "scheduleValue", "input_2")
model.load()

heating_setpoint = tb.ScheduleSystem(
    weekDayRulesetDict={
        "ruleset_default_value": 18.0,
        "ruleset_start_minute": [0], "ruleset_end_minute": [0],
        "ruleset_start_hour": [8], "ruleset_end_hour": [17],
        "ruleset_value": [21.0],
    },
    id="HeatingSetpoint",
)
cooling_setpoint = const(26.0, "CoolingSetpoint")
print("Model loaded:", len(model.components), "components")

In [ ]:
simulator = tb.Simulator(model)
optimizer = tb.Optimizer(simulator)

start = datetime.datetime(2024, 1, 4, tzinfo=tz.gettz("Europe/Copenhagen"))
end = start + datetime.timedelta(days=1)
step_size = 2400  # 40 min -> 36 decision variables over the day

t0 = time.perf_counter()
optimizer.optimize(
    start_time=start,
    end_time=end,
    step_size=step_size,
    variables=[(waterflow, "scheduleValue", 0, mf)],
    objectives=[
        (space_heater, "Power", "min"),
        (costs, "output", "min"),
    ],
    ineq_cons=[
        (building_space, "indoorTemperature", "upper", cooling_setpoint),
        (building_space, "indoorTemperature", "lower", heating_setpoint),
    ],
    method=("scipy", "SLSQP", "ad"),
    options={"maxiter": 10},
)
wall = time.perf_counter() - t0
print(f"\nTotal wall time: {wall:.1f} s for 10 SLSQP iterations "
      f"(36 decision variables, 36-step horizon, B=1)")

## Part B — Kernel benchmarks, CPU vs GPU

Shapes mirror a realistic MPC problem: `n_x = 6` states, `n_u = 8` inputs of which
one is the decision schedule, horizon `T = 96` steps (24 h at 15 min — a typical
day-ahead problem, longer than Part A's to make the gradient work meaningful).
`B` is the number of *simultaneous* rollouts: price/weather scenarios in stochastic
MPC, random restarts of the schedule search, or independent zones/buildings
optimized at once.

In [ ]:
# --- Benchmark helpers -------------------------------------------------------
N_X, N_U = 6, 8
T = 96               # 24 h at 15-minute steps (day-ahead horizon)
DT = 900.0
BATCHES = [1, 8, 64, 512, 4096]

def bench(fn, device, repeats=5):
    fn()  # warm-up
    if device == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

def make_problem(B, device, seed=0):
    """B rollout instances: discrete dynamics, exogenous inputs, prices."""
    g = torch.Generator().manual_seed(seed)
    A = torch.rand(B, N_X, N_X, generator=g) * 1e-5
    A = A - torch.diag_embed(A.sum(-1) + 1.0 / 3600.0)
    Bm = torch.rand(B, N_X, N_U, generator=g) * 1e-4
    Ad = torch.matrix_exp(A.to(device) * DT)
    Bd = torch.einsum(
        "bij,bjk->bik",
        Ad - torch.eye(N_X, device=device),
        torch.linalg.solve(A.to(device), Bm.to(device)),
    )
    u_exo = torch.rand(B, T, N_U - 1, generator=g).to(device)   # weather etc.
    price = (0.5 + 2.0 * torch.rand(B, T, generator=g)).to(device)
    x0 = torch.rand(B, N_X, generator=g).to(device)
    return Ad, Bd, u_exo, price, x0

def rollout_loss(schedule, Ad, Bd, u_exo, price, x0):
    """Forward rollout + cost/comfort loss; differentiable in the schedule."""
    x = x0
    loss = 0.0
    for t in range(T):
        u_t = torch.cat([u_exo[:, t], schedule[:, t : t + 1]], dim=-1)
        x = torch.einsum("bij,bj->bi", Ad, x) + torch.einsum("bij,bj->bi", Bd, u_t)
        power = 2000.0 * schedule[:, t]
        comfort = torch.relu(20.0 - x[:, 0]) ** 2       # soft comfort band
        loss = loss + (price[:, t] * power + 1e3 * comfort).sum()
    return loss

def run_table(name, runner, batches=BATCHES):
    rows = []
    for B in batches:
        row = {"B": B}
        for dev in DEVICES:
            row[dev] = bench(lambda: runner(B, dev), dev)
        if "cuda" in row:
            row["speedup"] = row["cpu"] / row["cuda"]
        rows.append(row)
    df = pd.DataFrame(rows).set_index("B")
    print(f"\n=== {name} (seconds per call) ===")
    print(df.to_string(float_format=lambda v: f"{v:.4f}"))
    return df

### O1 — Gradient through the rollout

One optimizer iteration: forward rollout of the horizon, then reverse-mode autodiff
of the loss w.r.t. the full decision schedule (`T` values per instance). This is
exactly the cost profile of the fast control objective's
value-plus-Jacobian evaluation inside SLSQP.

In [ ]:
def o1_gradient_through_rollout(B, device):
    Ad, Bd, u_exo, price, x0 = make_problem(B, device)
    schedule = torch.full((B, T), 0.3, device=device, requires_grad=True)
    loss = rollout_loss(schedule, Ad, Bd, u_exo, price, x0)
    loss.backward()
    return schedule.grad

df_o1 = run_table("O1: gradient through rollout (value + full Jacobian)",
                  o1_gradient_through_rollout)

### O2 — Batched gradient-descent schedule search

A full optimization *loop* rather than a single evaluation: 50 Adam steps on the
schedule, batched over `B` instances at once. This is the shape of population-based
or multi-restart schedule optimization (and of stochastic MPC, where `B` scenarios
share one schedule — here each instance keeps its own to maximize parallel work).

In [ ]:
N_ADAM = 50

def o2_batched_schedule_search(B, device):
    Ad, Bd, u_exo, price, x0 = make_problem(B, device)
    schedule = torch.full((B, T), 0.3, device=device, requires_grad=True)
    opt = torch.optim.Adam([schedule], lr=0.05)
    for _ in range(N_ADAM):
        opt.zero_grad()
        loss = rollout_loss(schedule.clamp(0.0, 1.0), Ad, Bd, u_exo, price, x0)
        loss.backward()
        opt.step()
    return schedule.detach()

# Smaller batches: this kernel is 50x heavier than a single evaluation.
df_o2 = run_table(f"O2: batched schedule search ({N_ADAM} Adam steps)",
                  o2_batched_schedule_search, batches=[1, 8, 64, 512])

In [ ]:
# --- Summary plot ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (df, title) in zip(
    axes,
    [
        (df_o1, "O1: gradient through rollout"),
        (df_o2, f"O2: schedule search ({N_ADAM} Adam steps)"),
    ],
):
    for dev in DEVICES:
        ax.loglog(df.index, df[dev] / df.index, "o-", label=dev)
    ax.set_title(title)
    ax.set_xlabel("batch size B (scenarios / restarts)")
    ax.grid(True, which="both", alpha=0.3)
axes[0].set_ylabel("seconds per instance")
axes[0].legend()
plt.tight_layout()
plt.show()

if "cuda" in DEVICES:
    print("GPU speedup at largest batch:")
    for name, df in [("O1", df_o1), ("O2", df_o2)]:
        print(f"  {name}: {df['speedup'].iloc[-1]:.1f}x "
              f"(B=1: {df['speedup'].iloc[0]:.2f}x)")

## How to read the results

Expected pattern (Colab T4; your numbers above will vary):

- **`B = 1` — a single MPC solve, as the optimizer runs today — favors the CPU.**
  The rollout is sequential in time and each step is a tiny operation on a 6-state
  system; GPU kernel-launch latency dominates and there is nothing to parallelize.

- **Per-instance cost on GPU drops roughly linearly with `B`** until saturation.
  Robust/stochastic MPC with hundreds of scenarios, multi-restart searches of
  non-convex schedule landscapes, or portfolio-scale optimization (many buildings,
  one GPU) are the workloads where an order of magnitude or more is available.

- **O2 shows the end-to-end effect**: when the whole optimization loop lives on the
  device, there is no per-iteration transfer overhead and the batched speedup
  carries through to the full solve.

**What this means for Twin4Build:** the single-solve MPC loop should stay on CPU.
The GPU opportunity is a *batched* optimizer API — one schedule optimized against
`B` uncertainty scenarios, or `B` independent zones/buildings solved simultaneously —
built on the same composed one-step map the fast objective already uses (its
`n_s`/`n_c` batch dimensions map directly onto `B`). The prerequisite is the same
device plumbing listed in the estimation notebook.